1. Create a bronze table that ingests raw sales data as-is (no transformations), preserving all original
columns plus an ingestion timestamp.

In [0]:
%python
from pyspark.sql.functions import *

df = spark.read.csv("/Volumes/dev/bronze/raw/sales.csv", header=True, inferSchema=True)
bronze_df = (df.withColumn("ingestion_timestamp", current_timestamp()))

bronze_df.write.format("delta").mode("append").saveAsTable("cyntexa_dev.sales.sales_bronze")

2. Build a silver table from bronze that removes duplicates, fixes data types, and drops clearly invalid
rows.

In [0]:
%python 
from pyspark.sql.functions import *

df = spark.read.table("cyntexa_dev.sales.sales_bronze")

silver_df = (df
            .withColumn("order_id", col("order_id").cast("integer"))
            .withColumn("customer_id", col("customer_id").cast("integer"))
            .withColumn("transaction_id", col("transaction_id").cast("integer"))
            .withColumn("product_id", col("product_id").cast("integer"))
            .withColumn("quantity", col("quantity").cast("integer"))
            .withColumn("discount_amount", col("discount_amount").cast("double"))
            .withColumn("total_amount", col("total_amount").cast("double"))
            .withColumn("order_date", col("order_date").cast("date"))

            .dropDuplicates(["order_id"])
            .dropna(how="all")
            .drop("ingestion_timestamp")

            .filter(col("order_id").isNotNull())
            .filter(col("customer_id").isNotNull())
            .filter(col("transaction_id").isNotNull())
            .filter(col("product_id").isNotNull())
            .filter(col("quantity").isNotNull())
            .filter(col("discount_amount").isNotNull())
            .filter(col("total_amount").isNotNull())
            .filter(col("order_date").isNotNull())
        )

silver_df.write.format("delta").mode("overwrite").saveAsTable("cyntexa_dev.silver.sales_silver")


3. Build a gold table that aggregates silver into a business-ready view (e.g., daily revenue by store).

-> since i do not have table with store id's i make making the gold kpi similar - revenue by product 

In [0]:
create schema if not exists cyntexa_dev.gold;

In [0]:
CREATE VIEW IF NOT EXISTS cyntexa_dev.gold.product_gold AS
SELECT
product_id,
SUM(quantity) AS total_quantity,
SUM(total_amount) AS total_revenue
FROM
cyntexa_dev.silver.sales_silver
GROUP BY
product_id
ORDER BY
total_revenue DESC;